# Consolidated Plotting: Ensemble Performance & Feature Tornado

This notebook uses the consolidated functions from `function.py` to generate visualizations.

**Note**: This notebook expects the data files to be in the **parent directory** (`../`):
- `test_predictions.csv` (Lasso Predictions)
- `xgb_result.parquet` (XGBoost Predictions)
- `result_df.parquet` (NN Predictions)
- `feature_importance_final.csv` (Feature Importance Data)

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Import consolidated functions
# Since function.py is in the same directory
from function import *

# Ensure Chinese characters display correctly
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

## 1. Ensemble Performance Plot

In [ ]:
# Define Paths (Relative to Parent Directory)
root_path = ".."
lasso_path = os.path.join(root_path, "test_predictions.csv")
xgb_path = os.path.join(root_path, "xgb_result.parquet")
nn_path = os.path.join(root_path, "result_df.parquet")

try:
    required_files = {
        'Lasso': lasso_path,
        'XGB': xgb_path,
        'NN': nn_path
    }
    
    missing = []
    for name, path in required_files.items():
        if not os.path.exists(path):
            missing.append(f"{name}: {path}")
    
    if missing:
        print("Missing files in parent directory:")
        for f in missing:
            print(f"  - {f}")
        print("\nPlease ensure the data files exist in the parent folder.")
    else:
        # 1. Load predictions
        print("Loading model predictions...")
        df_lasso = load_prediction(lasso_path, 'Lasso')
        df_xgb = load_prediction(xgb_path, 'XGB')
        df_nn = load_prediction(nn_path, 'NN')

        # 2. Align Data
        if len(df_lasso) != len(df_xgb):
            print(f"CRITICAL: Lasso ({len(df_lasso)}) and XGB ({len(df_xgb)}) lengths differ. Cannot infer Lasso index.")
        else:
            df_lasso.index = df_xgb.index
            
            # 3. Join all on index (Inner Join)
            print("Aligning dataframes by index...")
            
            df_all = df_xgb[['y', 'score_XGB']].rename(columns={'y': 'y_true'})
            df_all['score_Lasso'] = df_lasso['score_Lasso']
            
            # Merge NN
            df_nn_renamed = df_nn[['score_NN']]
            common_idx = df_all.index.intersection(df_nn.index)
            
            # Align NN to df_all
            df_combined = df_all.loc[common_idx].copy()
            df_combined['score_NN'] = df_nn.loc[common_idx, 'score_NN']
            
            print(f"Aligned samples: {len(df_combined)}")
            
            if len(df_combined) > 0:
                # 4. Prepare Ensemble Matrix
                y_true = df_combined['y_true'].values
                X_preds = np.column_stack([
                    df_combined['score_Lasso'].values,
                    df_combined['score_XGB'].values,
                    df_combined['score_NN'].values
                ])
                
                model_names = ['Lasso', 'XGB', 'NN']
                
                # 5. Optimize Weights
                print("\nOptimizing Ensemble Weights...")
                weights = optimize_weights(X_preds, y_true)
                
                print("\nOptimal Weights:")
                for name, w in zip(model_names, weights):
                    print(f"{name}: {w:.4f}")
                    
                # 6. Ensemble Performance
                final_pred = np.dot(X_preds, weights)
                
                # 7. Plot
                # Save assets to ../assets if exists, else ..
                assets_dir = os.path.join(root_path, "assets")
                if not os.path.exists(assets_dir):
                     assets_dir = root_path
                
                output_plot_path = os.path.join(assets_dir, "ensemble_performance.png")
                plot_performance(y_true, final_pred, "Ensemble Model", output_plot_path)

except Exception as e:
    print(f"Error: {e}")

## 2. Feature Tornado Plot

In [ ]:
# Load Data (Relative to Parent)
root_path = ".."
feature_file = os.path.join(root_path, "feature_importance_final.csv")

if os.path.exists(feature_file):
    print(f"Loading {feature_file}...")
    df_features = pd.read_csv(feature_file)
    
    assets_dir = os.path.join(root_path, "assets")
    if not os.path.exists(assets_dir):
            assets_dir = root_path
            
    output_tornado_path = os.path.join(assets_dir, "feature_tornado_final.png")
    plot_tornado(df_features, output_tornado_path)
else:
    print(f"File not found: {feature_file}")
    print("Please ensure 'feature_importance_final.csv' is in the parent folder.")